# Symbolic Derivation of the 1D Particle in a Box

This notebook derives the analytical solution of the one-dimensional particle in a box directly from the time-independent Schrödinger equation.

We start from the differential equation for a free quantum particle confined to a finite interval and show how the boundary conditions produce discrete allowed states. This makes the particle-in-a-box problem useful as both a physical model and a benchmark for numerical eigensolvers.

This notebook derives the analytical solution of the one-dimensional particle in a box directly from the Schrödinger equation.



Inside the box, the potential is taken to be zero. The walls are represented by homogeneous Dirichlet boundary conditions,
$$\begin{gather}
\psi(0)=0 \\
\psi(L)=0
\end{gather}$$

so the quantum state must vanish at both endpoints. The Hamiltonian in the interior is therefore the kinetic-energy operator,

$$
\hat{H}
=
-\frac{\hbar^2}{2m}
\frac{d^2}{dx^2}
$$

and the stationary Schrödinger equation is

$$
\hat{H}\psi(x)
=
E\psi(x)
$$

or equivalently,

$$
-\frac{\hbar^2}{2m}
\frac{d^2\psi}{dx^2}
=
E\psi
$$

This is an eigenvalue problem for a differential operator. The eigenfunctions are the allowed stationary states, and the eigenvalues are the corresponding allowed energies.

The main point of this notebook is that the discrete spectrum does not come from the differential equation alone. The differential equation gives a family of sinusoidal solutions. The boundary conditions select only those solutions that fit inside the box. This produces the quantized wave numbers

$$
k_n
=
\frac{n\pi}{L},
\qquad
n=1,2,3,\dots
$$

and the energy spectrum

$$
E_n
=
\frac{\hbar^2 k_n^2}{2m}
=
\frac{\hbar^2\pi^2 n^2}{2mL^2}
$$

The resulting normalized eigenfunctions are

$$
\psi_n(x)
=
\sqrt{\frac{2}{L}}
\sin\left(
\frac{n\pi x}{L}
\right)
$$

This symbolic derivation is also useful for validating the finite-difference solver. The numerical Hamiltonian approximates the same operator on a discrete grid. Its lowest eigenvalues and eigenvectors should converge to the analytical expressions derived here as the grid is refined.

We begin with the time-dependent Schrödinger equation (TDSE), because this is the dynamical equation for a quantum state. For a particle of mass $m$ moving in one spatial dimension under a potential $V(x)$, the TDSE is

$$
i\hbar \frac{\partial}{\partial t} \Psi(x,t)
=
\hat{H}\Psi(x,t)
$$


In [2]:
# physkit/qm/models/particle_in_box.py

import sympy as sp
class ParticleInABox1DSymbolic:
    """
    Symbolic expressions for the 1D infinite square well.

    This class is for derivation, display, and code generation.
    It is not the numerical eigensolver.
    """

    def __init__(self):
        self.x = sp.Symbol("x", real=True)
        self.x_min = sp.Symbol("x_min", real=True)
        self.L = sp.Symbol("L", positive=True, real=True)
        self.n = sp.Symbol("n", positive=True, integer=True)
        self.m = sp.Symbol("m", positive=True, real=True)
        self.hbar = sp.Symbol("hbar", positive=True, real=True)

    def k_n(self):
        return self.n * sp.pi / self.L

    def energy_n(self):
        k = self.k_n()
        return self.hbar**2 * k**2 / (2 * self.m)

    def psi_n(self):
        return sp.sqrt(sp.Rational(2, 1) / self.L) * sp.sin(
            self.n * sp.pi * (self.x - self.x_min) / self.L
        )

    def hamiltonian_on_psi(self):
        psi = self.psi_n()
        return -self.hbar**2 / (2 * self.m) * sp.diff(psi, self.x, 2)

    def verify_eigenvalue_equation(self):
        psi = self.psi_n()
        E = self.energy_n()
        Hpsi = self.hamiltonian_on_psi()

        return sp.simplify(Hpsi - E * psi)

In [ ]:
sym = ParticleInABox1DSymbolic()
print("psi_n =")
sp.pprint(sym.psi_n())

print("E_n =")
sp.pprint(sym.energy_n())

print("H psi - E psi =")
sp.pprint(sym.verify_eigenvalue_equation())

psi_n =
      ⎛π⋅n⋅(x - xₘᵢₙ)⎞
√2⋅sin⎜──────────────⎟
      ⎝      L       ⎠
──────────────────────
          √L          
E_n =
 2  2  2
π ⋅h̅ ⋅n 
────────
    2   
 2⋅L ⋅m 
H psi - E psi =
0


In [6]:
import sympy as sp
from IPython.display import display, Math

expr = sym.psi_n()

display(Math("\psi_n(x) =" +sp.latex(expr)))

<IPython.core.display.Math object>

In [10]:
# derive_particle_in_box_1d_symbolic.py

import sympy as sp
from IPython.display import display, Math, Markdown


# -----------------------------------------------------------------------------
# Display helpers
# -----------------------------------------------------------------------------

def show_heading(text: str) -> None:
    display(Markdown(f"## {text}"))


def show_plain(text: str) -> None:
    display(Markdown(text))


def show_math(expr) -> None:
    display(Math(sp.latex(expr)))


def show_eq(lhs: str, rhs) -> None:
    display(Math(lhs + " = " + sp.latex(rhs)))


def show_relation(lhs, rhs) -> None:
    display(Math(sp.latex(sp.Eq(lhs, rhs))))


# -----------------------------------------------------------------------------
# Symbols
# -----------------------------------------------------------------------------

x = sp.Symbol("x", real=True)
L = sp.Symbol("L", positive=True, real=True)
m = sp.Symbol("m", positive=True, real=True)
hbar = sp.Symbol("hbar", positive=True, real=True)
E = sp.Symbol("E", positive=True, real=True)
k = sp.Symbol("k", positive=True, real=True)
n = sp.Symbol("n", positive=True, integer=True)

psi = sp.Function("psi")


# -----------------------------------------------------------------------------
# 1. TISE
# -----------------------------------------------------------------------------

show_heading("1. Time-independent Schrödinger equation")

tise = sp.Eq(
    -hbar**2 / (2*m) * sp.diff(psi(x), x, 2),
    E * psi(x)
)

show_math(tise)


# -----------------------------------------------------------------------------
# 2. Standard ODE form
# -----------------------------------------------------------------------------

show_heading("2. Standard ODE form")

ode_E = sp.Eq(
    sp.diff(psi(x), x, 2) + (2*m*E/hbar**2) * psi(x),
    0
)

show_math(ode_E)

k_definition = sp.Eq(k**2, 2*m*E/hbar**2)
show_math(k_definition)

ode_k = sp.Eq(
    sp.diff(psi(x), x, 2) + k**2 * psi(x),
    0
)

show_math(ode_k)


# -----------------------------------------------------------------------------
# 3. General solution
# -----------------------------------------------------------------------------

show_heading("3. General solution of the ODE")

general_solution = sp.dsolve(ode_k, psi(x))
show_math(general_solution)

C1, C2 = sp.symbols("C_1 C_2")

psi_general = C1 * sp.sin(k*x) + C2 * sp.cos(k*x)

show_eq(r"\psi(x)", psi_general)


# -----------------------------------------------------------------------------
# 4. Apply left boundary condition
# -----------------------------------------------------------------------------

show_heading("4. Boundary condition at x = 0")

bc_left = sp.Eq(psi_general.subs(x, 0), 0)
show_math(bc_left)

C2_solution = sp.solve(bc_left, C2)[0]

show_eq(r"C_2", C2_solution)

psi_left = psi_general.subs(C2, C2_solution)

show_eq(r"\psi(x)", psi_left)


# -----------------------------------------------------------------------------
# 5. Apply right boundary condition
# -----------------------------------------------------------------------------

show_heading("5. Boundary condition at x = L")

bc_right = sp.Eq(psi_left.subs(x, L), 0)
show_math(bc_right)

show_plain("For a nonzero eigenstate, C_1 is not zero. Therefore the remaining condition is:")

show_eq(r"\sin(kL)", 0)

show_plain("The allowed wave numbers are:")

k_n = n * sp.pi / L

show_eq(r"k_n", k_n)


# -----------------------------------------------------------------------------
# 6. Energy eigenvalues
# -----------------------------------------------------------------------------

show_heading("6. Energy eigenvalues")

E_from_k = hbar**2 * k**2 / (2*m)

show_eq(r"E", E_from_k)

E_n = sp.simplify(E_from_k.subs(k, k_n))

show_eq(r"E_n", E_n)


# -----------------------------------------------------------------------------
# 7. Unnormalized eigenfunction
# -----------------------------------------------------------------------------

show_heading("7. Unnormalized eigenfunction")

A = sp.Symbol("A", positive=True, real=True)

psi_n_unnormalized = A * sp.sin(k_n * x)

show_eq(r"\psi_n(x)", psi_n_unnormalized)


# -----------------------------------------------------------------------------
# 8. Normalize
# -----------------------------------------------------------------------------

show_heading("8. Normalize the eigenfunction")

norm_integral = sp.integrate(
    psi_n_unnormalized**2,
    (x, 0, L)
)

show_eq(
    r"\int_0^L \psi_n(x)^2\,dx",
    norm_integral
)

A_solution = sp.solve(
    sp.Eq(norm_integral, 1),
    A
)[0]

show_eq(r"A", A_solution)

psi_n = sp.simplify(psi_n_unnormalized.subs(A, A_solution))

show_eq(r"\psi_n(x)", psi_n)


# -----------------------------------------------------------------------------
# 9. Verify eigenvalue equation
# -----------------------------------------------------------------------------

show_heading("9. Verify the eigenvalue equation")

H_psi_n = sp.simplify(
    -hbar**2/(2*m) * sp.diff(psi_n, x, 2)
)

E_psi_n = sp.simplify(E_n * psi_n)

residual = sp.simplify(H_psi_n - E_psi_n)

show_eq(r"\hat{H}\psi_n", H_psi_n)
show_eq(r"E_n\psi_n", E_psi_n)
show_eq(r"\hat{H}\psi_n - E_n\psi_n", residual)


# -----------------------------------------------------------------------------
# 10. Lambdify numerical expressions
# -----------------------------------------------------------------------------

show_heading("10. Convert symbolic expressions to numerical functions")

psi_n_func = sp.lambdify(
    (x, L, n),
    psi_n,
    "numpy"
)

E_n_func = sp.lambdify(
    (L, n, hbar, m),
    E_n,
    "numpy"
)

show_plain("Created NumPy-callable functions: `psi_n_func` and `E_n_func`.")

## 1. Time-independent Schrödinger equation

<IPython.core.display.Math object>

## 2. Standard ODE form

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 10. Convert symbolic expressions to numerical functions

Created NumPy-callable functions: `psi_n_func` and `E_n_func`.